In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
import json as js
import meshio as mio
import subprocess as sup
from scipy.interpolate import RBFInterpolator
import igl
from tqdm import tqdm
import pickle

In [9]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)

In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
path = "/Users/teseo/data/cell/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"
data_path = "/Users/teseo/data/cell/Embryogram test/centers.pkl"
# path = "/Users/zoeli/Documents/UVic/masters/other/fish/still.hdf5"
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/neweststill.hdf5"

#o ct 16
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/not_control.hdf5"

#o ct 10
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/realone.hdf5"

# new 
# path = "/Users/zoeli/Documents/UVic/masters/other/fish/20250523.hdf5"

# new 
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/poor_quality.hdf5"
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/20250702.hdf5"

#polyfem = "/Users/teseo/Documents/scuola/polyfem/polyfem.nosync/bin_rel.nosync/PolyFEM_bin"

In [16]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

# print(top, bottom, middle)
print(middle.shape)

(6076,)


In [27]:
centers = hdf5_file["bc_func/centers"][:].astype(float)
eps = hdf5_file["bc_func/eps"][()]

with open(data_path, "rb") as f:
    pickle.dump(centers, f)


NameError: name 'pickle' is not defined

In [13]:
E = hdf5_file["problem/E"][()] .astype(float)
nu = hdf5_file["problem/nu"][()] .astype(float)
is_linear = hdf5_file["problem/is_linear"][()] .astype(bool)

# E

In [18]:
fmiddle = set()
ftop = set()
fbottom = set()

for i in tqdm(range(T.shape[0])):
    tet = T[i, :]
    faces = [[tet[0], tet[1], tet[2]], [tet[0], tet[1], tet[3]], [tet[0], tet[2], tet[3]], [tet[1], tet[2], tet[3]]]
    for face in faces:
        if all(v in middle for v in face):
            fmiddle.add(tuple(np.sort(face)))
        elif all(v in top for v in face):
            ftop.add(tuple(np.sort(face)))
        elif all(v in bottom for v in face):
            fbottom.add(tuple(np.sort(face)))

100%|██████████| 888729/888729 [00:24<00:00, 36224.03it/s]


In [23]:
default_json = {
"geometry": {
        "mesh": "___",
        "volume_selection": 1,
        "surface_selection": {
            "file": "bc.txt",
            "boundary_only": False
        }
    },
"boundary_conditions": {
    "dirichlet_boundary": [
            {
                "id": 10,
                "value": [
                    {
                        "file_name": "rbf.py",
                        "function_name": "rbf_x"
                    },
                    {
                        "file_name": "rbf.py",
                        "function_name": "rbf_y"
                    },
                    {
                        "file_name": "rbf.py",
                        "function_name": "rbf_z"
                    },
                ]
            },
            {
                "id": 20,
                "value": [0, 0, 0]
            },
            {
                "id": 30,
                "value": [0, 0, 0]
            }
        ]
},
"materials": {
        "E": E,
        "id": 1,
        "nu": nu,
        "type": "LinearElasticity" if is_linear else "NeoHookean"
},
"output": {
    "json": "sim.json",
    "directory": "___",
    "paraview": {
        "file_name": "sim.vtu",
        "surface": True,
        "options": {
            "material": True,
            "forces": True
        },
        "vismesh_rel_area": 10000000
    }
}
}

In [24]:
def generate_json(out, V, T, fmiddle, fbottom, ftop):
    with open(f"{out}bc.txt", "w") as f:
        for face in fmiddle:
            f.write(f"10 {face[0]} {face[1]} {face[2]}\n")
        for face in fbottom:
            f.write(f"20 {face[0]} {face[1]} {face[2]}\n")
        for face in ftop:
            f.write(f"30 {face[0]} {face[1]} {face[2]}\n")

    mesh = mio.Mesh(points=V, cells={"tetra": T})
    mesh.write(f"{out}mesh.msh", file_format="gmsh")

    json = default_json.copy()
    json["geometry"]["mesh"] = f"mesh.msh"
    json["output"]["directory"] = out

    with open(f"{out}run.json", "w") as f:
        js.dump(json, f, indent=4)



generate_json("outnz/", V,T, fmiddle, fbottom, ftop)

In [ ]:
index=200
sup.run([polyfem, "-j", f"outnz/run_{index}.json"], check=True)

In [ ]:
rbf_disps = []

for i, disp in enumerate(disps):
    corrected_tf = [val < 0.5 for val in corrected[i]]
    rbf = RBFInterpolator(centers[corrected_tf], new_corrected_disps[i][corrected_tf])
    rbf_disp = rbf(centers)
    rbf_disps.append(rbf_disp)

In [ ]:
pl = pv.Plotter()

pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = centers.shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

#corrected_tf = [val < 0.5 for val in corrected[0]]
#rbf = RBFInterpolator(centers[corrected_tf], new_disps[0][corrected_tf])
#rbf_disp = rbf(centers)

vertices = np.vstack([centers, centers + rbf_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='green')

pl.camera_position = 'xz'
pl.camera.elevation = 25
def callback(x):
    #corrected_tf = [val < 0.5 for val in corrected[x]]
    #rbf = RBFInterpolator(centers[corrected_tf], new_disps[x][corrected_tf])
    #rbf_disp = rbf(centers)

    vertices = np.vstack([centers, centers + rbf_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='green')
    
    pl.update()
    #pl.camera_position = 'xy'

pl.show()
interact(callback, x=(0, len(disps)-1, 1))

In [ ]:
cells = [("vertex", np.array([[i,] for i in range(len(centers))]))]

for i, disp in enumerate(disps):
    mesh = mio.Mesh(centers, cells, point_data={
        "original_disps": disps[i], 
        "outlier_identification": corrected[i], 
        "corrected_pca_scale_translation": new_corrected_disps[i], 
        "corrected_scale_translation": new_corrected_disps_only_scale_and_translation[i], 
        "corrected_pca": new_corrected_disps_only_pca[i], 
       # "corrected_pca_scale_translation_with_smoothing": new_corrected_smoothed_disps[i], 
        "rbf": rbf_disps[i]
    })
    path = "/Users/zoeli/Documents/UVic/masters/other/fish/vtuFiles/disps" + str(i) + ".vtu"
    mesh.write(path)

In [ ]:
pl = pv.Plotter(notebook=False, off_screen=True)
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = centers.shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + rbf_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='green')

actor = pl.add_text(str(0), position='upper_right')
pl.camera_position = 'xz'
pl.camera.elevation = 25

# Open a gif
pl.open_gif("updated.gif")

# Update Z and write a frame for each updated position
nframe = 15
for index in range(len(rbf_disps)):
    vertices = np.vstack([centers, centers + rbf_disps[index]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='green')

    actor.set_text('upper_right', str(index))

    # Write a frame. This triggers a render.
    pl.write_frame()

# Closes and finalizes movie
pl.close()

In [ ]:
pl = pv.Plotter(notebook=False, off_screen=True)
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

actor = pl.add_text(str(0), position='upper_right')
pl.camera_position = 'xz'
pl.camera.elevation = 45

# Open a gif
pl.open_gif("test.gif")

# Update Z and write a frame for each updated position
nframe = 15
for index in range(len(rbf_disps)):
    pl.add_mesh(to_pyvista_mesh(centers + rbf_disps[index]), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

    actor.set_text('upper_right', str(index))

    # Write a frame. This triggers a render.
    pl.write_frame()

# Closes and finalizes movie
pl.close()

In [ ]:
pvsm = "C:/Users/zoeli/Documents/UVic/masters/other/fish/state.pvsm"
mesh = "C:/Users/zoeli/Documents/UVic/masters/other/fish/wildtype.obj"

for i in range(1, 4):
    sim = "C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim" + str(i) + ".vtu"
    LoadState(pvsm, filenames=[{"name": "sim110.vtu", "FileName": sim}, {"name": "wildtype.obj", "FileName": mesh}])
    my_source = FindSource("ResampleWithDataset1")
    SaveData("C:/Users/zoeli/Documents/UVic/masters/other/fish/outputest/sim" + str(i) + ".vtu", proxy=my_source)
    ResetSession()

C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim1.vtu
C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim2.vtu


In [285]:
tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim100.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim100.vtu")

In [286]:
tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim101.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim101.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim102.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim102.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim103.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim103.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim104.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim104.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim105.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim105.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim106.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim106.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim107.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim107.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim108.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim108.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim109.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim109.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim110.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim110.vtu")